# **SNU AI Challenge Baseline Code**

이 코드는 SNU AI Challenge 경진대회의 베이스라인 코드입니다.

## **Task**

주어진 스토리라인(캡션)에 맞게 4개의 이미지 프레임을 올바른 순서로 재배열하는 문제를 해결해야 합니다.

- **Input**: 자연어 문장과 여러 장의 프레임으로 구성된 데이터
  - 예: `{ "text": "자연어 문장", "frames": [image_3, image_1, image_4, image_2] }`
- **Output**: 정답 순서대로 다시 배열하였을 때 각 프레임의 위치
  - 예: `[3, 4, 1, 2]`
  - 정답 순서대로 다시 배열하였을 때 첫 번째 프레임은 3번째에 위치, 두 번째 프레임은 4번째에 위치, …

## **Baseline**

`Qwen2-VL-2B-Instruct` 모델을 사용

## **Overall Pipeline**

| 단계 | 내용 |
|------|------|
| Step 1 | 데이터 로드 (test.csv + 이미지) |
| Step 2 | 모델 로드 (Qwen2-VL-2B-Instruct from HuggingFace) |
| Step 3 | 추론 진행 (프롬프트 → 모델 생성 → 출력 파싱) |
| Step 4 | 결과 저장 (submission.csv) |


## 0. 환경 설정 및 패키지 설치


In [ ]:
# 필요한 패키지 설치
# 이미 설치되어 있으면 대부분 바로 넘어갑니다.
%pip install -q transformers pandas pillow tqdm scikit-learn accelerate

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 1. 라이브러리 import

import ast
import itertools
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor, Trainer, TrainingArguments


def show_df(df, n=5):
    """VSCode Interactive에서는 display, 일반 Python 실행에서는 print를 사용합니다."""
    try:
        from IPython.display import display
        display(df.head(n))
    except Exception:
        print(df.head(n))

In [ ]:
# 2. GPU 및 경로 설정

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU를 사용할 수 없습니다. "
        "VSCode 커널, Anaconda 환경, PyTorch CUDA 설치를 확인하세요."
    )

device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True

print("device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)

# 네 VSCode 폴더 구조 기준
PROJECT_DIR = r"C:\Users\lyh30\Documents\SNU_2026"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")

TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

# 제출 파일과 체크포인트 저장 위치
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.csv")

required_paths = [
    TRAIN_CSV,
    TEST_CSV,
    TRAIN_IMAGE_DIR,
    TEST_IMAGE_DIR,
]

for path in required_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(f"경로를 찾을 수 없습니다: {path}")

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_CSV:", TEST_CSV)
print("TRAIN_IMAGE_DIR:", TRAIN_IMAGE_DIR)
print("TEST_IMAGE_DIR:", TEST_IMAGE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SUBMISSION_PATH:", SUBMISSION_PATH)

device: cuda
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
CUDA: 13.0
PyTorch: 2.12.1+cu130
PROJECT_DIR: C:\Users\lyh30\Documents\SNU_2026
DATA_DIR: C:\Users\lyh30\Documents\SNU_2026\data
TRAIN_CSV: C:\Users\lyh30\Documents\SNU_2026\data\train.csv
TEST_CSV: C:\Users\lyh30\Documents\SNU_2026\data\test.csv
TRAIN_IMAGE_DIR: C:\Users\lyh30\Documents\SNU_2026\data\train
TEST_IMAGE_DIR: C:\Users\lyh30\Documents\SNU_2026\data\test
OUTPUT_DIR: C:\Users\lyh30\Documents\SNU_2026\data\outputs
SUBMISSION_PATH: C:\Users\lyh30\Documents\SNU_2026\data\outputs\submission.csv


In [ ]:
# 3. CSV 데이터 로드

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_CSV)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("sample_submission shape:", sample_submission.shape)
print("train columns:", train_df.columns.tolist())
print("test columns:", test_df.columns.tolist())

if "No_ordering" in train_df.columns:
    print("No_ordering counts:")
    print(train_df["No_ordering"].value_counts(dropna=False))

show_df(train_df)
show_df(test_df)

train shape: (9535, 8)
test shape: (819, 6)
sample_submission shape: (819, 2)
train columns: ['Id', 'Input_1', 'Input_2', 'Input_3', 'Input_4', 'Sentence', 'Answer', 'No_ordering']
test columns: ['Id', 'Input_1', 'Input_2', 'Input_3', 'Input_4', 'Sentence']
No_ordering counts:
No_ordering
False    8057
True     1478
Name: count, dtype: int64


,Id,Input_1,Input_2,Input_3,Input_4,Sentence,Answer,No_ordering
0,diZi5g,diZi5g_ggm.jpg,diZi5g_jfd.jpg,diZi5g_nva.jpg,diZi5g_udj.jpg,The camera pans left to reveal two people knee...,"[1, 2, 3, 4]",True
1,u7w0lr,u7w0lr_aot.jpg,u7w0lr_shc.jpg,u7w0lr_uyv.jpg,u7w0lr_vuj.jpg,A girl hula hoops indoors before the scene shi...,"[3, 1, 2, 4]",False
2,2vqGOF,2vqGOF_buy.jpg,2vqGOF_nvf.jpg,2vqGOF_rlp.jpg,2vqGOF_vad.jpg,The woman lowers her gaze to apply a yellow co...,"[1, 2, 3, 4]",True
3,kSE41E,kSE41E_mug.jpg,kSE41E_ndb.jpg,kSE41E_nkm.jpg,kSE41E_vpx.jpg,"The man moves closer to the mirror, tilting hi...","[3, 2, 4, 1]",False
4,oO5UVE,oO5UVE_aat.jpg,oO5UVE_iox.jpg,oO5UVE_mkz.jpg,oO5UVE_qqu.jpg,The cookie sheet with dough moves closer to th...,"[3, 1, 2, 4]",False


,Id,Input_1,Input_2,Input_3,Input_4,Sentence
0,WI8o3F,WI8o3F_bia.jpg,WI8o3F_tuc.jpg,WI8o3F_vyx.jpg,WI8o3F_zcv.jpg,A person on an inner tube slides down a snowy ...
1,Ae7zLm,Ae7zLm_iis.jpg,Ae7zLm_otu.jpg,Ae7zLm_vqz.jpg,Ae7zLm_wru.jpg,The camera moves from a close-up of colorful n...
2,4QSIII,4QSIII_ayw.jpg,4QSIII_jqa.jpg,4QSIII_mqo.jpg,4QSIII_sto.jpg,The players on the beach court move forward an...
3,k7h4gt,k7h4gt_cuk.jpg,k7h4gt_ixd.jpg,k7h4gt_lqh.jpg,k7h4gt_rvh.jpg,The violin shifts from horizontal to vertical ...
4,ZtTwCx,ZtTwCx_npt.jpg,ZtTwCx_slc.jpg,ZtTwCx_whe.jpg,ZtTwCx_zve.jpg,"The person speaks to the camera, then lifts pa..."


## 2. 데이터 로드 & 모델 로드

In [ ]:
# 4. CLIP 모델 로드

MODEL_NAME = "laion/CLIP-ViT-B-32-laion2B-s34B-b79K"

print("Loading processor:", MODEL_NAME)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

print("Loading model:", MODEL_NAME)
clip_model = CLIPModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to(device)

clip_model.eval()

print("model device:", next(clip_model.parameters()).device)

Loading processor: laion/CLIP-ViT-B-32-laion2B-s34B-b79K
Loading model: laion/CLIP-ViT-B-32-laion2B-s34B-b79K


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model device: cuda:0


In [ ]:
# 5. 정답 라벨 변환

# 4장의 순서는 총 4! = 24가지입니다.
ORDERS = list(itertools.permutations([1, 2, 3, 4]))
ORDER_TO_CLASS = {order: idx for idx, order in enumerate(ORDERS)}
CLASS_TO_ORDER = {idx: list(order) for idx, order in enumerate(ORDERS)}


def parse_answer_to_class(answer):
    """Answer 값 예: '[1, 2, 3, 4]'를 0~23 class id로 바꿉니다."""
    if isinstance(answer, str):
        answer = ast.literal_eval(answer)

    answer = tuple(int(x) for x in answer)

    if answer not in ORDER_TO_CLASS:
        raise ValueError(f"Invalid answer: {answer}")

    return ORDER_TO_CLASS[answer]


def class_to_answer(pred_class):
    """예측 class id를 제출용 Answer 리스트로 바꿉니다."""
    return CLASS_TO_ORDER[int(pred_class)]


train_df["label_class"] = train_df["Answer"].apply(parse_answer_to_class)
print(train_df["label_class"].value_counts().sort_index())

label_class
0     1478
1      352
2      359
3      346
4      342
5      355
6      343
7      365
8      305
9      364
10     346
11     344
12     358
13     338
14     360
15     343
16     386
17     349
18     319
19     334
20     359
21     358
22     366
23     366
Name: count, dtype: int64


In [ ]:
# 6. Dataset 및 collate 함수

class CLIPOrderingDataset(Dataset):
    """문장 1개와 이미지 4장을 입력으로 받아 순서 class를 학습하는 Dataset입니다."""

    def __init__(self, df, image_dir, is_train=True):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_files = [
            row["Input_1"],
            row["Input_2"],
            row["Input_3"],
            row["Input_4"],
        ]

        images = []
        for img_file in img_files:
            img_path = os.path.join(self.image_dir, str(row["Id"]), str(img_file))

            if not os.path.exists(img_path):
                raise FileNotFoundError(f"Image not found: {img_path}")

            images.append(Image.open(img_path).convert("RGB"))

        item = {
            "id": row["Id"],
            "sentence": str(row["Sentence"]),
            "images": images,
        }

        if self.is_train:
            item["labels"] = parse_answer_to_class(row["Answer"])

        return item


def build_batch(batch, include_ids=False):
    """Dataset 샘플 여러 개를 CLIP 모델 입력 tensor로 묶습니다."""
    texts = [item["sentence"] for item in batch]

    # batch size가 B면 이미지는 B * 4개입니다.
    all_images = []
    for item in batch:
        all_images.extend(item["images"])

    text_inputs = processor(
        text=texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    )

    image_inputs = processor(
        images=all_images,
        return_tensors="pt",
    )

    # [B * 4, 3, H, W] -> [B, 4, 3, H, W]
    pixel_values = image_inputs["pixel_values"]
    pixel_values = pixel_values.view(len(batch), 4, *pixel_values.shape[1:])

    result = {
        "input_ids": text_inputs["input_ids"],
        "attention_mask": text_inputs["attention_mask"],
        "pixel_values": pixel_values,
    }

    if "labels" in batch[0]:
        result["labels"] = torch.tensor(
            [item["labels"] for item in batch],
            dtype=torch.long,
        )

    if include_ids:
        result["ids"] = [item["id"] for item in batch]

    return result


def train_collate_fn(batch):
    return build_batch(batch, include_ids=False)


def test_collate_fn(batch):
    return build_batch(batch, include_ids=True)

In [ ]:
# 7. CLIP ordering 모델 및 평가 함수

class CLIPOrderingModel(nn.Module):
    """CLIP은 고정하고, 위에 작은 24-class 순서 분류기만 학습합니다."""

    def __init__(self, clip_model, hidden_size=512, num_classes=24):
        super().__init__()
        self.clip = clip_model

        # 첫 실행 안정성을 위해 CLIP 본체는 freeze합니다.
        for param in self.clip.parameters():
            param.requires_grad = False

        # 문장 embedding 1개 + 이미지 embedding 4개 = 총 5개 embedding
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 5, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids=None, attention_mask=None, pixel_values=None, labels=None):
        batch_size = input_ids.shape[0]

        # CLIP은 freeze했으므로 gradient 계산을 하지 않습니다.
        with torch.no_grad():
            text_outputs = self.clip.text_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            text_embeds = self.clip.text_projection(text_outputs.pooler_output)

            flat_pixel_values = pixel_values.view(
                batch_size * 4,
                pixel_values.shape[2],
                pixel_values.shape[3],
                pixel_values.shape[4],
            )

            image_outputs = self.clip.vision_model(pixel_values=flat_pixel_values)
            image_embeds = self.clip.visual_projection(image_outputs.pooler_output)

        image_embeds = image_embeds.view(batch_size, 4, -1)

        text_embeds = F.normalize(text_embeds, dim=-1)
        image_embeds = F.normalize(image_embeds, dim=-1)

        features = torch.cat(
            [
                text_embeds,
                image_embeds[:, 0],
                image_embeds[:, 1],
                image_embeds[:, 2],
                image_embeds[:, 3],
            ],
            dim=-1,
        )

        logits = self.classifier(features)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return {"loss": loss, "logits": logits}


def compute_metrics(eval_pred):
    """검증 accuracy를 계산합니다."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = (preds == labels).mean()
    return {"accuracy": accuracy}

In [ ]:
# 8. 학습/검증 데이터 분리

# 먼저 작은 subset으로 정상 실행을 확인합니다.
# 전체 학습을 할 때는 둘 다 None으로 바꾸세요.
TRAIN_LIMIT = 1000
VALID_LIMIT = 300

# No_ordering은 순서가 애매한 샘플이므로 첫 학습에서는 제외합니다.
if "No_ordering" in train_df.columns:
    no_ordering_mask = train_df["No_ordering"].astype(str).str.lower().isin(
        ["true", "1", "yes"]
    )
    train_used = train_df[~no_ordering_mask].copy()
else:
    train_used = train_df.copy()

stratify_col = train_used["label_class"]
if stratify_col.value_counts().min() < 2:
    stratify_col = None

train_part, valid_part = train_test_split(
    train_used,
    test_size=0.2,
    random_state=42,
    stratify=stratify_col,
)

if TRAIN_LIMIT is not None:
    train_part = train_part.head(TRAIN_LIMIT)

if VALID_LIMIT is not None:
    valid_part = valid_part.head(VALID_LIMIT)

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)

train_dataset = CLIPOrderingDataset(train_part, TRAIN_IMAGE_DIR, is_train=True)
valid_dataset = CLIPOrderingDataset(valid_part, TRAIN_IMAGE_DIR, is_train=True)

ordering_model = CLIPOrderingModel(clip_model).to(device)

train_part: (1000, 9)
valid_part: (300, 9)


In [25]:
# 9. Trainer 설정 및 학습

common_training_args = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=1e-4,
    logging_steps=20,
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    fp16=False,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none",
)

# transformers 버전에 따라 eval_strategy/evaluation_strategy 이름이 다를 수 있어 둘 다 대응합니다.
try:
    training_args = TrainingArguments(
        **common_training_args,
        eval_strategy="steps",
    )
except TypeError:
    training_args = TrainingArguments(
        **common_training_args,
        evaluation_strategy="steps",
    )

trainer = Trainer(
    model=ordering_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=train_collate_fn,
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy
100,3.079205,3.137519,0.046250


KeyboardInterrupt: 

In [16]:
# 10. 검증 성능 확인

eval_result = trainer.evaluate()
print(eval_result)

Training Loss,Validation Loss,Step,Accuracy
3.108127,3.136871,2250,0.051250


{'eval_loss': 3.1368706226348877, 'eval_accuracy': 0.05125}


In [17]:
# 11. 테스트 데이터셋 및 DataLoader 생성

test_dataset = CLIPOrderingDataset(test_df, TEST_IMAGE_DIR, is_train=False)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=test_collate_fn,
)

print("test batches:", len(test_loader))

test batches: 205


In [18]:
# 12. 테스트 추론

ordering_model.eval()
predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, total=len(test_loader)):
        batch_ids = batch.pop("ids")

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = ordering_model(**batch)
        pred_classes = outputs["logits"].argmax(dim=-1).cpu().tolist()

        for sample_id, pred_class in zip(batch_ids, pred_classes):
            predictions.append(
                {
                    "Id": sample_id,
                    "Answer": str(class_to_answer(pred_class)),
                }
            )

submission_df = pd.DataFrame(predictions)
print("raw predictions:", submission_df.shape)
show_df(submission_df)

  0%|          | 0/205 [00:00<?, ?it/s]

raw predictions: (819, 2)


,Id,Answer
0,WI8o3F,"[1, 3, 2, 4]"
1,Ae7zLm,"[2, 3, 4, 1]"
2,4QSIII,"[4, 3, 1, 2]"
3,k7h4gt,"[4, 3, 1, 2]"
4,ZtTwCx,"[1, 3, 2, 4]"


In [ ]:
# 13. 제출 파일 생성 및 저장

# sample_submission의 Id 순서에 맞춰 정렬합니다.
submission_df = sample_submission[["Id"]].merge(
    submission_df,
    on="Id",
    how="left",
)

# 혹시 누락된 예측이 있으면 기본값으로 채웁니다.
submission_df["Answer"] = submission_df["Answer"].fillna(str([1, 2, 3, 4]))

submission_df.to_csv(SUBMISSION_PATH, index=False)

print("saved:", SUBMISSION_PATH)
print("submission shape:", submission_df.shape)
show_df(submission_df)

OSError: Cannot save file into a non-existent directory: 'C:\Users\lyh30\Documents\SNU_2026\data\outputs'

LoRA 학습된 모델로 추론

## 4. 결과 저장 (Submission)

### 제출 파일 형식 (submission.csv)

제출 파일은 반드시 아래 형식을 따라야 합니다:

| 컬럼 | 타입 | 설명 | 예시 |
|------|------|------|------|
| `Id` | string | 테스트 샘플 고유 ID | `WI8o3F` |
| `Answer` | string | 정답 순서대로 다시 배열하였을 때 각 프레임의 위치 (Python 리스트 형태의 문자열) | `[3, 1, 4, 2]` |

#### 올바른 예시

```csv
Id,Answer
WI8o3F,[3, 1, 4, 2]
Ae7zLm,[1, 2, 3, 4]
...


In [ ]:
# 14. 최종 확인

print("Submission file:", SUBMISSION_PATH)
print("Exists:", os.path.exists(SUBMISSION_PATH))

if os.path.exists(SUBMISSION_PATH):
    check_df = pd.read_csv(SUBMISSION_PATH)
    print(check_df.shape)
    show_df(check_df)